# Machine Learning Assignment 1 — VAR 2
## Polynomial Regression — BT2024129

This notebook follows the same workflow/style as the supplied VAR 1 notebook, adapted for **VAR 2**.

- Polynomial degree: **1 to 20**
- 5-fold shuffled cross-validation
- Ridge degree/alpha search
- Fine Ridge alpha search
- Degree refinement
- Lasso degree/alpha search
- Final test prediction file: `BT2024129_pred_var2.csv`


In [38]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error
from google.colab import files


## 1. Upload VAR 2 training data

In [39]:
uploaded = files.upload()


Saving BT2024129_train_var2.csv to BT2024129_train_var2 (2).csv


In [40]:
df = pd.read_csv('BT2024129_train_var2.csv')

X = df.drop(columns=['y'])
y = df['y']

print(df.head())
print('\nShape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nX shape:', X.shape)
print('y shape:', y.shape)


         x1        x2        x3          y
0  0.405068 -0.024034  0.285641   0.680972
1 -1.000000  0.131738 -0.495196  -1.872695
2  0.859808 -0.214529 -0.804103   2.680131
3 -0.798160 -0.352199 -0.583345   0.775998
4  1.000000  1.000000 -0.543765  21.344758

Shape: (1000, 4)
Columns: ['x1', 'x2', 'x3', 'y']

X shape: (1000, 3)
y shape: (1000,)


In [41]:
print('Missing values:')
print(df.isnull().sum())


Missing values:
x1    0
x2    0
x3    0
y     0
dtype: int64


## 2. Ridge polynomial regression search

In [42]:
pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])


In [43]:
param_grid = {
    'poly__degree': list(range(1, 21)),
    'ridge__alpha': [0, 1e-6, 1e-5, 1e-4, 1e-3,
                     1e-2, 1e-1, 1, 10, 100]
}

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [44]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X, y)


GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('ridge', Ridge())]),
             n_jobs=-1,
             param_grid={'poly__degree': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                          13, 14, 15, 16, 17, 18, 19, 20],
                         'ridge__alpha': [0, 1e-06, 1e-05, 0.0001, 0.001, 0.01,
                                          0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

In [45]:
print('Best parameters:')
print(grid_search.best_params_)
print('\nBest CV MSE:', -grid_search.best_score_)


Best parameters:
{'poly__degree': 12, 'ridge__alpha': 1}

Best CV MSE: 0.2592459889358805


In [46]:
best_model = grid_search.best_estimator_

y_pred_train = best_model.predict(X)
train_mse = mean_squared_error(y, y_pred_train)

print('Training MSE:', train_mse)


Training MSE: 0.1619987720379938


## 3. Fine Ridge alpha search

In [47]:
fine_alphas = np.linspace(0.1, 5, 21)

fine_param_grid = {
    'poly__degree': [grid_search.best_params_['poly__degree']],
    'ridge__alpha': fine_alphas
}

fine_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=fine_param_grid,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

fine_grid.fit(X, y)

print('Best parameters:', fine_grid.best_params_)
print('Best CV MSE:', -fine_grid.best_score_)


Best parameters: {'poly__degree': 12, 'ridge__alpha': np.float64(1.8150000000000002)}
Best CV MSE: 0.25667748667342216


## 4. Second fine Ridge alpha search

In [48]:
best_alpha = fine_grid.best_params_['ridge__alpha']
best_degree = fine_grid.best_params_['poly__degree']

fine_alphas_2 = np.linspace(
    max(0, best_alpha - 1),
    best_alpha + 1,
    41
)

fine_param_grid_2 = {
    'poly__degree': [best_degree],
    'ridge__alpha': fine_alphas_2
}

fine_grid_2 = GridSearchCV(
    pipeline,
    fine_param_grid_2,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

fine_grid_2.fit(X, y)

print('Best parameters:', fine_grid_2.best_params_)
print('Best CV MSE:', -fine_grid_2.best_score_)


Best parameters: {'poly__degree': 12, 'ridge__alpha': np.float64(1.8650000000000002)}
Best CV MSE: 0.2566735106269674


## 5. Final Ridge degree search (degrees 1–20)

In [49]:
best_alpha = fine_grid_2.best_params_['ridge__alpha']

param_grid_degree = {
    'poly__degree': range(1, 21),
    'ridge__alpha': [best_alpha]
}

degree_search = GridSearchCV(
    pipeline,
    param_grid_degree,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

degree_search.fit(X, y)

print('Best parameters:', degree_search.best_params_)
print('Best CV MSE:', -degree_search.best_score_)


Best parameters: {'poly__degree': 12, 'ridge__alpha': np.float64(1.8650000000000002)}
Best CV MSE: 0.2566735106269674


In [50]:
best_model = degree_search.best_estimator_

y_pred_train = best_model.predict(X)
train_mse = mean_squared_error(y, y_pred_train)

print('Training MSE:', train_mse)
print('CV MSE:', -degree_search.best_score_)


Training MSE: 0.167592081302817
CV MSE: 0.2566735106269674


## 6. Lasso polynomial regression search

In [51]:
lasso_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('lasso', Lasso(max_iter=10000))
])

lasso_grid = {
    'poly__degree': range(4, 14),
    'lasso__alpha': np.logspace(-4, -2, 5)
}

lasso_search = GridSearchCV(
    lasso_pipeline,
    lasso_grid,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

lasso_search.fit(X, y)


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.900e+00, tolerance: 4.823e+00
  model = cd_fast.enet_coordinate_descent(


GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('lasso', Lasso(max_iter=10000))]),
             n_jobs=-1,
             param_grid={'lasso__alpha': array([0.0001    , 0.00031623, 0.001     , 0.00316228, 0.01      ]),
                         'poly__degree': range(4, 14)},
             scoring='neg_mean_squared_error')

In [52]:
print('Best Lasso parameters:')
print(lasso_search.best_params_)
print('\nBest Lasso CV MSE:', -lasso_search.best_score_)


Best Lasso parameters:
{'lasso__alpha': np.float64(0.00031622776601683794), 'poly__degree': 10}

Best Lasso CV MSE: 0.2531166842562361


## 7. Load VAR 2 test data and generate predictions

In [53]:
uploaded2 = files.upload()


Saving BT2024129_test_var2.csv to BT2024129_test_var2 (2).csv


In [54]:
test_df = pd.read_csv('BT2024129_test_var2.csv')
X_test = test_df.copy()

print('Test shape:', X_test.shape)
print(X_test.head())


Test shape: (1000, 3)
         x1        x2        x3
0 -0.185273 -0.809004  1.000000
1  0.983879  0.069062 -0.252585
2 -0.135800  0.547913 -0.220012
3 -0.902964 -1.000000 -1.000000
4 -0.429827  0.995460  0.140823


In [55]:
y_test_pred = lasso_search.best_estimator_.predict(X_test)

print('Number of predictions:', len(y_test_pred))
print('\nFirst 10 predictions:')
print(y_test_pred[:10])


Number of predictions: 1000

First 10 predictions:
[-2.24591891  4.24185744  0.04114727 -0.23882522  5.90550527 21.2397007
  2.26605543  3.05448954 -6.60118752  4.90320309]


In [56]:
from sklearn.linear_model import ElasticNet

In [57]:
elastic_pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("elastic", ElasticNet(
        max_iter=3000,
        tol=1e-3,
        selection="random"
    ))
])

elastic_param_grid = {
    "poly__degree": [10, 11, 12, 13, 14],
    "elastic__alpha": [0.01, 0.03, 0.1, 0.3],
    "elastic__l1_ratio": [0.2, 0.5, 0.8]
}

elastic_search = GridSearchCV(
    estimator=elastic_pipeline,
    param_grid=elastic_param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

elastic_search.fit(X, y)

elastic_cv_mse = -elastic_search.best_score_

print("Best parameters:", elastic_search.best_params_)
print("Elastic Net CV MSE:", elastic_cv_mse)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best parameters: {'elastic__alpha': 0.01, 'elastic__l1_ratio': 0.2, 'poly__degree': 14}
Elastic Net CV MSE: 0.286701796741912


In [58]:
print("Ridge Training R²:", fine_grid_2.best_estimator_.score(X, y))

print("Lasso Training R²:", lasso_search.best_estimator_.score(X, y))

print("Elastic Net Training R²:", elastic_search.best_estimator_.score(X, y))

Ridge Training R²: 0.9965254591053511
Lasso Training R²: 0.9965100580369831
Elastic Net Training R²: 0.9956341262191319


## 8. Save final VAR 2 prediction file

In [59]:
output = pd.DataFrame({
    'y': y_test_pred
})

output.to_csv('BT2024129_pred_var2.csv', index=False)

print('Saved: BT2024129_pred_var2.csv')


Saved: BT2024129_pred_var2.csv


In [60]:
submission = pd.read_csv('BT2024129_pred_var2.csv')

print(submission.head())
print('\nShape:', submission.shape)
print('\nColumns:', submission.columns.tolist())
print('\nMissing values:')
print(submission.isnull().sum())

assert submission.columns.tolist() == ['y']
assert len(submission) == len(test_df)
assert submission['y'].isnull().sum() == 0

print('\nPrediction file is valid.')

          y
0 -2.245919
1  4.241857
2  0.041147
3 -0.238825
4  5.905505

Shape: (1000, 1)

Columns: ['y']

Missing values:
y    0
dtype: int64

Prediction file is valid.


In [61]:
files.download('BT2024129_pred_var2.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>